[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_7_continuous_observation_spaces_complete.ipynb)

<div style="text-align:center">
    <h1>
        Continuous state spaces
    </h1>
</div>

<br>

<div style="text-align:center">
    In this notebook we will learn how to adapt tabular methods to continuous state spaces. We will do it with two methods:
    state aggregation and tile coding.
</div>

<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_policy, plot_tabular_cost_to_go, plot_stats, seed_everything


## Import the necessary software libraries:

In [ ]:
# Random module: used by the epsilon-greedy policy to pick exploratory actions.
import random
# Gym: provides the MountainCar environment and the observation-wrapper base class.
import gym
# NumPy: powers the Q-value table and the bucketing math (linspace, digitize).
import numpy as np
# Matplotlib: drawing library used by the course plotting helpers.
import matplotlib.pyplot as plt
# tqdm: shows a progress bar while we loop over thousands of training episodes.
from tqdm import tqdm

## Implement state aggregation


### Create the environment

In [ ]:
# Build the classic MountainCar control task. Its state is CONTINUOUS:
# (car position, car velocity) are real numbers, not a finite set of grid cells.
env = gym.make('MountainCar-v0')
# Fix all random seeds (Python, NumPy, the env) so the run is reproducible.
seed_everything(env)

### Create the state aggregation wrapper

### Why tabular methods fail on continuous state spaces

Every method so far (Monte Carlo, SARSA, Q-Learning, n-step) stored one number per state-action pair in a **table**. That works when states are a finite, countable set - like the 25 cells of a maze.

MountainCar is different. Its state is a pair of **real numbers**: the car's position and its velocity. Between any two positions there are *infinitely many* others, so:

- We **cannot** build a table with one row per state - there are uncountably many.
- Even if we rounded, we'd almost never visit the exact same state twice, so most table entries would stay at their initial value forever - the agent would never learn.

The fix in this notebook is **discretization**: chop each continuous dimension into a finite number of **buckets** (bins) and treat every state that falls in the same bucket as "the same state." This is called **state aggregation**. Once the world is carved into a finite grid, our familiar tabular SARSA works again, unchanged.

The wrapper below uses `np.linspace` to build the bucket **edges** and `np.digitize` to find which bucket a continuous value lands in.

In [ ]:
# A gym ObservationWrapper lets us transform every observation the env returns
# BEFORE the agent sees it. Here we convert the continuous (position, velocity)
# into a pair of discrete bucket indices so a normal Q-table can store values.
class StateAggregationEnv(gym.ObservationWrapper):

    def __init__(self, env, bins, low, high):
        # Wrap the underlying continuous environment.
        super().__init__(env)
        # For each state dimension, build the bucket EDGES with linspace.
        # `l-1` edges create `l` buckets along that dimension.
        self.buckets = [np.linspace(j,k, l-1) for j,k,l in zip(low, high, bins)]
        # Advertise the new, DISCRETE observation space (a grid of bins x bins cells).
        self.observation_space = gym.spaces.MultiDiscrete(nvec=bins.tolist())

    def observation(self, obs):
        # np.digitize finds which bucket each continuous value falls into.
        # The result is a tuple of integer indices that can index a Q-table.
        indices = tuple(np.digitize(i, b) for i,b in zip(obs, self.buckets))
        return indices

In [ ]:
# Choose a 20 x 20 grid: 20 buckets for position, 20 for velocity.
bins = np.array([20, 20])
# Read the smallest and largest legal value of each state dimension from the env.
low = env.observation_space.low
high = env.observation_space.high
# Wrap the environment so every observation becomes a (row, col) bucket pair.
saenv = StateAggregationEnv(env, bins=bins, low=low, high=high)

In [ ]:
# Inspect the bucket edges that define the discretization for each dimension.
saenv.buckets

### Compare the original environment to the one with aggregated states

In [ ]:
# Show the NEW discrete observation space and draw one random sample from it.
print(f"Modified observation space: {saenv.observation_space}, \n\
Sample state: {saenv.observation_space.sample()}")

In [ ]:
# Show the ORIGINAL continuous observation space for comparison (real-valued box).
print(f"Original observation space: {env.observation_space}, \n\
Sample state: {env.observation_space.sample()}")

### Create the $Q(s,a)$ value table

In [ ]:
# The Q-table: one value per (position-bucket, velocity-bucket, action).
# Shape 20 x 20 x 3 = 1200 numbers, all initialised to zero.
action_values = np.zeros((20,20, 3))

### Create the $\epsilon$-greedy policy: $\pi(s)$

In [ ]:
# Epsilon-greedy policy over the discretized state.
def policy(state, epsilon=0.):
    # With probability epsilon, explore: pick a random action (0, 1 or 2).
    if np.random.random() < epsilon:
        return np.random.randint(3)
    else:
        # Otherwise exploit: look up the 3 action-values for this bucket cell.
        av = action_values[state]
        # Pick the best action, breaking ties at random among equal maxima.
        return np.random.choice(np.flatnonzero(av == av.max()))

### Test the SARSA algorithm on the modified environment

**Tabular SARSA, unchanged.** Once the state is discretized into bucket indices, this is the *exact same* SARSA you already know. SARSA is **on-policy**: it picks the next action `a'` with the same epsilon-greedy policy it is learning, and updates toward `r + gamma * Q(s', a')`. The discretization wrapper is doing all the heavy lifting - it converts continuous coordinates into integer table indices, so a plain NumPy Q-table just works.

In [ ]:
# Tabular SARSA on the aggregated (discretized) MountainCar.
def sarsa(action_values, policy, episodes, alpha=0.1, gamma=0.99, epsilon=0.2):
    # Keep the total reward of each episode so we can plot learning progress.
    stats = {'Returns': []}
    for episode in tqdm(range(1, episodes + 1)):
        # Reset returns a discrete bucket pair (thanks to the wrapper).
        state = saenv.reset()
        # SARSA chooses the first action up front (on-policy).
        action = policy(state, epsilon)
        done = False
        ep_return = 0
        while not done:
            # Take the action; observe next bucket state, reward, and done flag.
            next_state, reward, done, _ = saenv.step(action)
            # SARSA picks the NEXT action from the same policy (this is the 'A' in SARSA).
            next_action = policy(next_state, epsilon)

            # Current estimate Q(s, a).
            qsa = action_values[state][action]
            # Estimate of the action we will actually take next: Q(s', a').
            next_qsa = action_values[next_state][next_action]
            # SARSA update: move Q(s,a) toward the TD target r + gamma * Q(s', a').
            action_values[state][action] = qsa + alpha * (reward + gamma * next_qsa - qsa)
            # Shift the (state, action) window forward by one step.
            state = next_state
            action = next_action
            ep_return += reward
        stats['Returns'].append(ep_return)
    return stats

In [ ]:
# Train for 20,000 episodes. epsilon=0 here means a purely greedy policy
# (exploration comes from the random tie-breaking and the noisy start states).
stats = sarsa(action_values, policy, 20000, alpha=0.1, epsilon=0.)

In [ ]:
# Plot the per-episode return so we can see learning improve over time.
plot_stats(stats)

### Plot the learned policy: $\pi(s)$

In [ ]:
# Overlay the learned greedy action on each grid cell of the state space.
# B = push Back/left, N = do Nothing, F = push Forward/right.
plot_policy(action_values, env.render(mode='rgb_array'), \
            action_meanings={0: 'B', 1: 'N', 2: 'F'})

### Plot the cost to go: $ - \max_a \hat q(s,a|\theta)$

In [ ]:
# Cost-to-go = -max_a Q(s,a): how far (in cost) the agent is from the goal.
plot_tabular_cost_to_go(action_values, xlabel="Car Position", ylabel="Velocity")

### Test the resulting policy

In [ ]:
# Run the greedy policy for 2 episodes and watch the car solve the hill.
test_agent(saenv, policy, 2)

In [ ]:
# Reset the environment to a fresh starting state.
saenv.reset()

<br><br><br><br>

## Implement Tile Coding


### Create the environment

In [ ]:
# Re-create a fresh MountainCar for the tile-coding experiment.
env = gym.make('MountainCar-v0')
# Re-seed for reproducibility.
seed_everything(env)

### Create the Tile Coding wrapper

### Why tile coding beats a single coarse grid

State aggregation gave us a working agent, but it has a weakness: every point inside a bucket looks **identical**, and points just across a bucket edge look **completely different**. A coarse grid throws away fine detail; a fine grid needs a huge table and learns slowly.

**Tile coding** is the clever fix. Instead of one grid we lay down several grids (called *tilings*), each **shifted by a fraction of a cell**. A continuous state now activates one tile per tiling, and we represent its value as the **average across all the tilings**. Because the tilings are offset, two nearby points share *most* but not *all* of their tiles - so the agent can tell them apart while still generalizing between them. The result is a smoother, more accurate value function at a modest extra cost (here, 4 tables instead of 1).

In [ ]:
# Tile coding uses SEVERAL overlapping grids ('tilings'), each shifted a little.
# A continuous point falls into one tile per tiling; averaging across the shifted
# grids gives smoother generalization than a single coarse grid (state aggregation).
class TileCodingEnv(gym.ObservationWrapper):

    def __init__(self, env, bins, low, high, n=4):
        super().__init__(env)
        # Build `n` shifted tilings up front.
        self.tilings = self._create_tilings(bins, high, low, n)

    def observation(self, obs):
        # For each tiling, find which tile the observation lands in.
        indices = []
        for t in self.tilings:
            # Digitize each dimension against this tiling's bucket edges.
            tiling_indices = tuple(np.digitize(i, b) for i,b in zip(obs, t))
            indices.append(tiling_indices)
        # Return one bucket-pair per tiling (a list of n index tuples).
        return indices

    def _create_tilings(self, bins, high, low, n):
        # Asymmetric displacement pattern (1, 3, 5, ...) recommended for tile coding.
        displacement_vector = np.arange(1,2*len(bins),2)
        tilings = []
        for i in range(1, n + 1):
            # Randomly widen the low/high bounds a touch so tilings differ.
            low_i = low - random.random() * .2 * low
            high_i = high + random.random() * .2 * high
            # Width of one bucket along each dimension for this tiling.
            segment_sizes = (high_i - low_i) / bins
            # Offset this tiling by a fraction of a bucket so grids don't align.
            displacements = displacement_vector * i % n
            displacements = displacements * (segment_sizes / n)
            low_i += displacements
            high_i += displacements
            # Bucket edges for this shifted tiling.
            buckets_i = [np.linspace(j,k, l-1) for j,k,l in zip(low_i, high_i, bins)]
            tilings.append(buckets_i)
        return tilings

In [ ]:
# Use 4 overlapping tilings, each a 20 x 20 grid.
tilings = 4
bins = np.array([20, 20])
low = env.observation_space.low
high = env.observation_space.high
# Wrap the env so each observation becomes a list of 4 bucket-pairs.
tcenv = TileCodingEnv(env, bins=bins, low=low, high=high, n=tilings)

### Compare the original environment to the one with aggregated states

In [ ]:
# Show the (still continuous) observation space plus one tile-coded sample.
print(f"Modified observation space: {tcenv.observation_space}, \n\
Sample state: {tcenv.reset()}")

In [ ]:
# Show the original continuous observation space and one raw sample.
print(f"Original observation space: {env.observation_space}, \n\
Sample state: {env.reset()}")

### Create the $Q(s,a)$ value table

In [ ]:
# Q-table now has a leading axis of size 4: one 20x20x3 table PER tiling.
action_values = np.zeros((4, 20, 20, 3))

### Create the $\epsilon$-greedy policy: $\pi(s)$

In [ ]:
# Epsilon-greedy policy for tile coding.
def policy(state, epsilon=0.):
    # Explore with probability epsilon.
    if np.random.random() < epsilon:
        return np.random.randint(3)
    else:
        # `state` is a list of bucket-pairs, one per tiling.
        av_list = []
        for i, idx in enumerate(state):
            # Look up the 3 action-values for this tiling's tile.
            av = action_values[i][idx]
            av_list.append(av)

        # Combine the tilings by AVERAGING their action-values (the key idea).
        av = np.mean(av_list, axis=0)
        # Greedy choice with random tie-breaking.
        return np.random.choice(np.flatnonzero(av==av.max()))

### Test the SARSA algorithm on the modified environment

**SARSA with multiple tilings.** Nothing about the SARSA *idea* changes - we still move each Q-value toward `r + gamma * Q(s', a')`. The only difference is bookkeeping: a single observation now activates one tile in *each* of the 4 tilings, so we apply the same TD update to all 4 tables. Because the policy reads the **average** of the tilings, learning is spread across them and the value surface comes out smoother than the blocky single-grid version above.

In [ ]:
# SARSA adapted for tile coding: update EACH tiling's table separately.
def sarsa(action_values, policy, episodes, alpha=0.1, gamma=0.99, epsilon=0.2):
    stats = {'Returns': []}
    for episode in tqdm(range(1, episodes + 1)):
        state = tcenv.reset()
        action = policy(state, epsilon)
        done = False
        ep_return = 0
        while not done:
            next_state, reward, done, _ = tcenv.step(action)
            next_action = policy(next_state, epsilon)

            # Loop over the tilings, pairing this state's tile with the next state's tile.
            for i, (idx, next_idx) in enumerate(zip(state, next_state)):
                # Current and next Q-value for THIS tiling.
                qsa = action_values[i][idx][action]
                next_qsa = action_values[i][next_idx][next_action]
                # Apply the SARSA TD update to this tiling's table.
                action_values[i][idx][action] = qsa + alpha * (reward + gamma * next_qsa - qsa)

            state = next_state
            action = next_action
            ep_return += reward
        stats['Returns'].append(ep_return)
    return stats

In [ ]:
# Train the tile-coding agent for 20,000 episodes.
stats = sarsa(action_values, policy, 20000, alpha=0.1, epsilon=0.)

In [ ]:
# Plot the learning curve (returns per episode).
plot_stats(stats)

### Plot the learned policy: $\pi(s)$

In [ ]:
# Average the 4 tilings into one table, then draw the greedy policy.
plot_policy(action_values.mean(axis=0), env.render(mode='rgb_array'), \
            action_meanings={0: 'B', 1: 'N', 2: 'F'})

### Plot the cost to go: $ - \max_a \hat q(s,a|\theta)$

In [ ]:
# Cost-to-go surface from the averaged tilings.
plot_tabular_cost_to_go(action_values.mean(axis=0), \
                        xlabel="Car Position", ylabel="Velocity")

### Test the resulting policy

In [ ]:
# Run the learned tile-coding policy for 2 test episodes.
test_agent(tcenv, policy, 2)

## Resources

[[1] Reinforcement Learning: An Introduction. Section 9.5.4: Tile Coding](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)